In [7]:
# use datacq kernel

import getpass
import hashlib
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import requests
from requests.structures import CaseInsensitiveDict

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)
FIXTURES = Path("fixtures")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

### Credentials Input


In [6]:
# Endpoints
TOKEN_URL = "https://id.twitch.tv/oauth2/token"
STREAMS_URL = "https://api.twitch.tv/helix/streams"
USERS_URL = "https://api.twitch.tv/helix/users"

# Credentials
CLIENT_ID = os.environ.get("TWITCH_CLIENT_ID")
CLIENT_SECRET = os.environ.get("TWITCH_CLIENT_SECRET")

if not CLIENT_ID:
    CLIENT_ID = getpass.getpass("Twitch Client ID: ")
if not CLIENT_SECRET:
    CLIENT_SECRET = getpass.getpass("Twitch Client Secret: ")

### The offline switch


In [8]:
OFFLINE = os.environ.get("OFFLINE") == "1"
POLITE_DELAY = 0.0 if OFFLINE else 1.0   # seconds between live requests


def cache_key(url, params=None):
    """A short, stable name for one (url, params) pair.
    """
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


class FixtureResponse:

    def __init__(self, record):
        self.url = record.get("final_url", record["url"])
        self.status_code = record["status_code"]
        self.headers = CaseInsensitiveDict(record["headers"])
        self._body = record["body"]

    def json(self):
        return self._body

    @property
    def links(self):
        header = self.headers.get("link")
        if not header:
            return {}
        return {l["rel"]: l for l in requests.utils.parse_header_links(header)}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"{self.status_code} for {self.url}")


def offline_get(url, params=None, **kwargs):
    path = FIXTURES / f"{cache_key(url, params)}.json"
    if not path.exists():
        raise RuntimeError(
            f"no fixture for {url} {params}")
    return FixtureResponse(json.loads(path.read_text()))

if OFFLINE:
    requests.get = offline_get
    print("OFFLINE:", len(list(FIXTURES.glob("*.json"))), "fixtures loaded")
else:
    print("live requests")

live requests


In [ ]:
TWITCH_HEADERS = {
    "Client-Id": cid,
    "Authorization": f"Bearer {token}",
}

### OAuth Token Exchange

In [9]:
def get_oauth_token(cid, secret):
  payload = {
      "client_id": cid,
      "client_secret": secret,
      "grant_type": "client_credentials",
  }
  r = requests.post(TOKEN_URL, data=payload, timeout=10)
  r.raise_for_status()
  return r.json()["access_token"]


TOKEN = get_oauth_token(CLIENT_ID, CLIENT_SECRET) if not OFFLINE else "mock_token"

HEADERS = {"Client-Id": CLIENT_ID, "Authorization": f"Bearer {TOKEN}"}

In [ ]:
def http_get(url, params=None, headers=HEADERS, attempts=5):
    for k in range(attempts):
        try:
            r = requests.get(url, params=params, headers=headers,
                             timeout=10)
        except requests.RequestException as exc:   # timeout, DNS, reset
            status, retry_after = repr(exc), 0
        else:
            # 200 -> return the parsed body
            if (r.status_code == 200):
                return r.json()

            # status that will not change -> raise_for_status()
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()


            # 3. otherwise remember the status, and read Ratelimit-Reset
            status = r.status_code
            retry_after = 0
            if "Ratelimit-Reset" in r.headers:
                try:    # this is a timestamp : we can derive wait time
                    reset_epoch = int(r.headers["Ratelimit-Reset"])
                    retry_after = max(0, reset_epoch - int(time.time()))
                except (ValueError, TypeError):
                    retry_after = 0


        if k + 1 < attempts:

            if retry_after > 0:
                wait = retry_after
            else:
                wait = random.uniform(0, 2**k)

            print(f"  {status} on attempt {k + 1}, sleeping {wait:.1f}s")
            time.sleep(wait)
            
    raise RuntimeError(f"gave up after {attempts} attempts: {url}")


In [11]:
requests_made = 0


def cached_get(url, params=None):
    path = CACHE / f"{cache_key(url, params)}.json"
    if path.exists():
        return json.loads(path.read_text())

    global requests_made
    requests_made += 1
    body = http_get(url, params)
    path.write_text(json.dumps(body))
    time.sleep(POLITE_DELAY)
    return body

### Pagination Loop